In [3]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from datetime import datetime, timezone
import matplotlib.pyplot as plt

from shapely.geometry import Polygon
import geopandas as gpd

import os
import sys
import glob

In [ ]:
current_dir = os.getcwd()
ROOT_DIR = os.path.abspath(os.path.join(current_dir, '..'))
DATA_EXT_DIR = os.path.join(ROOT_DIR, 'data', 'external')
DATA_PROC_DIR = os.path.join(ROOT_DIR, 'data', 'processed')
DATA_BASES_DIR = os.path.join(ROOT_DIR, 'data', 'processed', 'bases')
DATA_MAP_DIR = os.path.join(ROOT_DIR, 'data', 'processed', 'mapas')

dirs = [DATA_EXT_DIR, DATA_PROC_DIR, DATA_BASES_DIR, DATA_MAP_DIR]
for d in dirs:
    os.makedirs(d, exist_ok=True)

In [5]:
uf_ref = 'MG'

In [20]:
ns = {"cap": "urn:oasis:names:tc:emergency:cap:1.2"}
print("Baixando feed principal de avisos...")

url_feed = "https://apiprevmet3.inmet.gov.br/avisos/rss"
r = requests.get(url_feed, timeout=10)

if r.status_code != 200:
    print("Erro ao acessar feed:", r.status_code)
    sys.exit()

if not r.text.strip().startswith("<"):
    print("Resposta não é XML válido. Conteúdo recebido:")
    print(r.text[:200])
    sys.exit()

root = ET.fromstring(r.content)

print("Download feed principal de avisos conlcuido")

Baixando feed principal de avisos...
Download feed principal de avisos conlcuido


In [21]:
alertas_lista = []

for item in root.findall(".//item"):

    link = item.findtext("link")

    try:
        r_item = requests.get(link, timeout=5)

        if r_item.status_code != 200:
            continue

        root_item = ET.fromstring(r_item.content)

        ns = {"cap": "urn:oasis:names:tc:emergency:cap:1.2"}

        identifier = root_item.findtext("cap:identifier", namespaces=ns)
        sent = root_item.findtext("cap:sent", namespaces=ns)
        status = root_item.findtext("cap:status", namespaces=ns)
        msgType = root_item.findtext("cap:msgType", namespaces=ns)
        scope = root_item.findtext("cap:scope", namespaces=ns)

        category = root_item.findtext(".//cap:category", namespaces=ns) 
        evento = root_item.findtext(".//cap:event", namespaces=ns)       
        responseType = root_item.findtext(".//cap:responseType", namespaces=ns)
        urgency = root_item.findtext(".//cap:urgency", namespaces=ns)
        severity = root_item.findtext(".//cap:severity", namespaces=ns)
        certainty = root_item.findtext(".//cap:certainty", namespaces=ns)
        onset = root_item.findtext(".//cap:onset", namespaces=ns)
        expires = root_item.findtext(".//cap:expires", namespaces=ns) 
        senderName = root_item.findtext(".//cap:senderName", namespaces=ns)

        color = root_item.findtext(".//cap:parameter[cap:valueName='ColorRisk']/cap:value",  namespaces=ns)

        municipios_text = root_item.findtext(".//cap:parameter[cap:valueName='Municipios']/cap:value", namespaces=ns)

        if not municipios_text:
            continue

        onset_dt = datetime.fromisoformat(onset)
        expires_dt = datetime.fromisoformat(expires)  
        extracao_dt = datetime.now(timezone.utc).astimezone(onset_dt.tzinfo)

        municipios_lista = [m.strip() for m in municipios_text.split(",")]

        for item_mun in municipios_lista:

            try:
                nome = item_mun.split(" - ")[0].strip()
                uf = item_mun.split(" - ")[1].split(" ")[0].strip()
                cod_ibge = item_mun.split("(")[1].replace(")", "").strip()

                if uf == uf_ref:
                    alertas_lista.append({
                        "identifier": identifier,
                        "sent": sent,
                        "status": status,
                        "msgType": msgType,
                        "scope": scope,
                        "link": link,
                        "municipio": nome,
                        "cod_ibge": cod_ibge,
                        "category": category,
                        "evento": evento,
                        "responseType": responseType,
                        "urgency": urgency,
                        "severity": severity,
                        "certainty": certainty,
                        "onset": onset_dt,
                        "expires": expires_dt,
                        "senderName": senderName,
                        "color": color,
                        "dthr_extracao": extracao_dt
                    })

            except:
                continue

    except:
        continue

df_alertas_temp = pd.DataFrame(alertas_lista)

In [23]:
# A iteração nos poligonos foi feita separda porque demora mais, além disso os poligonos são os mesmo para um mesmo link

links_unicos = df_alertas_temp['link'].unique()

cache = {}
ns = {'cap': 'urn:oasis:names:tc:emergency:cap:1.2'}

for url in links_unicos:
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        root = ET.fromstring(response.content)
        polygon = root.find('.//cap:area/cap:polygon', ns)
        
        cache[url] = polygon.text if polygon is not None else None
    
    except:
        cache[url] = None

df_alertas_temp['polygon'] = df_alertas_temp['link'].map(cache)

In [24]:
df_alertas = df_alertas_temp.copy()
df_alertas = df_alertas.replace({'Moderate': 'Perigo Potencial',
                                 'Severe': 'Perigo',
                                 'Extreme': 'Grande Perigo'})

In [25]:
hoje = datetime.today().strftime('%Y%m%d')
nome_arquivo_variavel = f'inmet_{uf_ref}_{hoje}.csv'

caminho_completo_variavel = os.path.join(DATA_BASES_DIR, nome_arquivo_variavel)
df_alertas.to_csv(caminho_completo_variavel, index=False)
print(f"Arquivo salvo em: {caminho_completo_variavel}")

Arquivo salvo em: c:\Users\Usuário\Documents\Projects\pessoal\inmet_avisos_meteorologicos\data\processed\bases\inmet_MG_20260306.csv


In [26]:
url_ibge = f"https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2022/UFs/{uf_ref}/{uf_ref}_Municipios_2022.zip"

DATA_MAP_FILE = os.path.join(ROOT_DIR, 'data', 'external', f'{uf_ref}_Municipios_2022.zip')

if not os.path.exists(DATA_MAP_FILE):
    print("Baixando malha municipal para {uf_ref}...")
    
    r = requests.get(url_ibge)
    r.raise_for_status()  # garante erro se download falhar
    
    with open(DATA_MAP_FILE, "wb") as f:
        f.write(r.content)

gdf = gpd.read_file(f"zip://{DATA_MAP_FILE}")

In [27]:
#Gerando os mapas de maneira automática e salvando

gdf["CD_MUN"] = gdf["CD_MUN"].astype(str)
gdf = gdf.to_crs(epsg=4326)
uf_unido = gdf.geometry.union_all()

# manter apenas 1 linha por link
df_unico = (df_alertas.dropna(subset=['polygon', 'link']).loc[df_alertas['polygon'].str.strip() != ""].drop_duplicates(subset=['link']))
print("Total de links únicos hoje:", len(df_unico))

mapas_gerados = 0

for idx, row in df_unico.iterrows():
    try:
        poly_str = row['polygon']
        cor = row['color'] if pd.notna(row['color']) else "#FF0000"
        evento = row['evento']
        severidade = row['severity']
        onset = row['onset']
        expires = row['expires']

        try:
            onset_dt = pd.to_datetime(onset) # Data YYYYMMDD            
            onset_data_str = onset_dt.strftime('%Y%m%d')            
            onset_hora_str = onset_dt.strftime('%H%M') # Hora HHMM (seguro para nome de arquivo)

        except Exception:
            hoje = datetime.today()
            onset_data_str = hoje.strftime('%Y%m%d')
            onset_hora_str = hoje.strftime('%H%M')

        # Criar pasta do dia
        pasta_dia = os.path.join(DATA_MAP_DIR, f"mapas_{uf_ref}_{onset_data_str}")
        os.makedirs(pasta_dia, exist_ok=True)

        # Coordenadas do polígono
        coords = [
            (float(p.split(',')[1]), float(p.split(',')[0]))
            for p in poly_str.strip().split(" ")
            if "," in p
        ]

        if len(coords) < 3:
            continue

        polygon_geom = Polygon(coords)

        if not polygon_geom.is_valid:
            polygon_geom = polygon_geom.buffer(0)

        polygon_recortado = polygon_geom.intersection(uf_unido)

        if polygon_recortado.is_empty:
            continue

        gdf_alerta = gpd.GeoDataFrame(geometry=[polygon_recortado], crs="EPSG:4326")

        fig, ax = plt.subplots(figsize=(8, 8))

        gdf.plot(ax=ax, color='white', edgecolor='black', linewidth=0.3)

        gdf_alerta.plot(
            ax=ax,
            color=cor,
            alpha=0.6
        )

        titulo = (
            f"Aviso: {evento}\n"
            f"Severidade {severidade}\n"
            f"Data Hora Inicio: {onset}\n"
            f"Data Hora Fim: {expires}\n"
        )

        ax.set_title(titulo, fontsize=12)
        ax.axis("off")

        arquivo_mapa = f"inmet_{evento}_{severidade}_{onset_data_str}_{onset_hora_str}.png"
        caminho_arquivo = os.path.join(pasta_dia, arquivo_mapa)

        plt.savefig(
            caminho_arquivo,
            dpi=300,
            bbox_inches='tight'
        )

        plt.close()

        mapas_gerados += 1

    except Exception as e:
        print(f"Erro no alerta {idx}: {e}")

print("Mapas gerados com sucesso:", mapas_gerados)
print("Arquivos salvos em:", DATA_MAP_DIR)

Total de links únicos hoje: 22
Mapas gerados com sucesso: 22
Arquivos salvos em: c:\Users\Usuário\Documents\Projects\pessoal\inmet_avisos_meteorologicos\data\processed\mapas
